# メータの画像から矢印の角度を求める

## ライブラリのインストール
Colab.の場合は起動するたびに実行

Anacondaの場合は1度実行すれば以後実行する必要はない

In [ ]:
#!pip install tensorflow
#!pip install opencv-python

## ライブラリのインポート

In [1]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tensorflow.keras.layers import Input, Dense, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from PIL import Image
import glob
import os
import re

# tf.test.gpu_device_name() # Check for GPU

## データの読み込み

Colab.とAnacondaのどちらかを実行

Colab.の場合

In [ ]:
train_list = glob.glob('/content/drive/MyDrive/angle32x32/train/*/*.jpg')#Colab.の場合
valid_list = glob.glob('/content/drive/MyDrive/angle32x32/valid/*/*.jpg')
labels = sorted(os.listdir('/content/drive/MyDrive/angle32x32/train'), key=int)

print(labels) # ラベルの表示

['0', '30', '60', '90', '120', '150', '180', '210', '240', '270', '300', '330']


Anacondaの場合

In [4]:
train_list = glob.glob('angle32x32/train/*/*.jpg')#Windows+Anacondaの場合
valid_list = glob.glob('angle32x32/valid/*/*.jpg')
labels = sorted(os.listdir('angle32x32/train'), key=int)
print(labels) # ラベルの表示

['0', '30', '60', '90', '120', '150', '180', '210', '240', '270', '300', '330']


ラベルの作成

Colab.とAnacondaのどちらかを実行

Colab.の場合

In [ ]:
train_labels = [re.search(r'/(\d+)_', f).groups()[0] for f in train_list]#Colab.の場合
valid_labels = [re.search(r'/(\d+)_', f).groups()[0] for f in valid_list]

# for regression model training: convert label name to int32
train_labels_reg = [int(i) for i in train_labels]
valid_labels_reg = [int(i) for i in valid_labels]

Anacondaの場合

In [5]:
train_labels = [re.search(r'\\(\d+)_', f).groups()[0] for f in train_list]#Windows+Anacondaの場合
valid_labels = [re.search(r'\\(\d+)_', f).groups()[0] for f in valid_list]

# for regression model training: convert label name to int32
train_labels_reg = [int(i) for i in train_labels]
valid_labels_reg = [int(i) for i in valid_labels]

ファイルリストとラベルをtf.Tensor形式へ変換する

In [6]:
train_ds = tf.data.Dataset.from_tensor_slices((train_list, train_labels_reg))
valid_ds = tf.data.Dataset.from_tensor_slices((valid_list, valid_labels_reg))

# ファイルのオープンとデータロード，正規化をバッチを生成するたびに行うための関数
def load_and_conversion(paths, labels):
    x = []
    for f in paths:
        raw = tf.io.read_file(f)
        image = tf.image.decode_image(raw, channels=1)
#        image = tf.image.resize(image,(32,32)) #画像の大きさを変える場合
        x.append(image.numpy() / 255.0)  # 正規化
    return x, labels

#
# Dynamic Conversion from image file list to tf.Tensor format using tf.Data API
#
AUTOTUNE = tf.data.experimental.AUTOTUNE
train_ds = train_ds.repeat(1)
train_ds = train_ds.batch(12) # ミニバッチを作るがバッチトレーニングで行うため，全ての角度のデータを使う
train_ds = train_ds.map(lambda paths, labels: tf.py_function(load_and_conversion, [paths, labels], Tout=[tf.float32, tf.int32]),
                        num_parallel_calls=AUTOTUNE) # load image data and conversion
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
valid_ds = valid_ds.batch(12)
valid_ds = valid_ds.map(lambda paths, labels: tf.py_function(load_and_conversion, [paths, labels], Tout=[tf.float32, tf.int32]),
                        num_parallel_calls=AUTOTUNE) # load image data and conversion

In [7]:
next(iter(train_ds)) # バッチの生成確認用（実行しなくてもよい）

(<tf.Tensor: shape=(12, 32, 32, 1), dtype=float32, numpy=
 array([[[[0.        ],
          [0.        ],
          [0.        ],
          ...,
          [0.        ],
          [0.        ],
          [0.        ]],
 
         [[0.        ],
          [0.        ],
          [0.        ],
          ...,
          [0.        ],
          [0.        ],
          [0.        ]],
 
         [[0.        ],
          [0.        ],
          [0.        ],
          ...,
          [0.        ],
          [0.        ],
          [0.        ]],
 
         ...,
 
         [[0.        ],
          [0.        ],
          [0.        ],
          ...,
          [0.        ],
          [0.        ],
          [0.        ]],
 
         [[0.        ],
          [0.        ],
          [0.        ],
          ...,
          [0.        ],
          [0.        ],
          [0.        ]],
 
         [[0.        ],
          [0.        ],
          [0.        ],
          ...,
          [0.        ],
     

## ネットワークの登録
DNNネットワーク

In [8]:
def angle_model_DNN():
    input = Input(shape=(32, 32, 1), name='input')
    h = Flatten()(input)
    h = Dense(1024, activation='relu', name='dense_1')(h)
    h = Dense(1024, activation='relu', name='dense_2')(h)
    output = Dense(1, activation='linear', name='output')(h) # 出力は１つのノード
    return Model(inputs=input, outputs=output)

CNNネットワーク

In [9]:
def angle_model_CNN():
    input = Input(shape=(32, 32, 1), name='input')
    h = Conv2D(16, (3,3), activation='relu', padding='same')(input)
    h = MaxPooling2D((2, 2))(h)
    h = Conv2D(32, (3,3), activation='relu', padding='same')(h)
    h = MaxPooling2D((2, 2))(h)
    h = Conv2D(64, (3,3), activation='relu', padding='same')(h)
    h = MaxPooling2D((2, 2))(h)
    h = Flatten()(h)
    h = Dense(1024, activation='relu', name='dense_1')(h)
    output = Dense(1, activation='linear', name='output')(h) # 出力は１つのノード
    return Model(inputs=input, outputs=output)

In [10]:
model = angle_model_DNN()
#model = angle_model_CNN()
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input (InputLayer)          [(None, 32, 32, 1)]       0         
                                                                 
 flatten (Flatten)           (None, 1024)              0         
                                                                 
 dense_1 (Dense)             (None, 1024)              1049600   
                                                                 
 dense_2 (Dense)             (None, 1024)              1049600   
                                                                 
 output (Dense)              (None, 1)                 1025      
                                                                 
Total params: 2,100,225
Trainable params: 2,100,225
Non-trainable params: 0
_________________________________________________________________


## モデルの設定
最適化関数と損失関数の設定

In [11]:
model.compile(optimizer=Adam(), loss='mse', metrics=['mae'])

## 学習の開始

In [12]:
history = model.fit(train_ds, epochs=500, validation_data=valid_ds)

Epoch 1/500
1/1 [==============================] - 4s 4s/step - loss: 37938.0078 - mae: 164.9417 - val_loss: 35986.8750 - val_mae: 159.9445
Epoch 2/500
1/1 [==============================] - 0s 90ms/step - loss: 35986.8750 - mae: 159.9445 - val_loss: 33730.7500 - val_mae: 153.8910
Epoch 3/500
1/1 [==============================] - 0s 102ms/step - loss: 33730.7461 - mae: 153.8910 - val_loss: 30738.2344 - val_mae: 145.3607
Epoch 4/500
1/1 [==============================] - 0s 101ms/step - loss: 30738.2344 - mae: 145.3607 - val_loss: 26993.9199 - val_mae: 134.9151
Epoch 5/500
1/1 [==============================] - 0s 90ms/step - loss: 26993.9199 - mae: 134.9151 - val_loss: 22658.7031 - val_mae: 122.5785
Epoch 6/500
1/1 [==============================] - 0s 93ms/step - loss: 22658.7031 - mae: 122.5785 - val_loss: 18086.7363 - val_mae: 109.7688
Epoch 7/500
1/1 [==============================] - 0s 102ms/step - loss: 18086.7363 - mae: 109.7688 - val_loss: 13855.1494 - val_mae: 98.2532
Epoch 

## テストデータを用いた予測

Colab.の場合

In [18]:
test_list = glob.glob('/content/drive/MyDrive/angle32x32/test2/*.jpg')#Colab.の場合
for f in test_list:
    raw = tf.io.read_file(f)
    img = tf.image.decode_image(raw, channels=1)
    img = img.numpy() / 255.0
    img = tf.expand_dims(img, axis=0) # バッチ次元の追加
    pred = model.predict(img) # 推論
    print(f, f'推定角度 {pred[0][0]:.2f}度')

1/1 [==============================] - 0s 14ms/step
/content/drive/MyDrive/angle32x32/test2/300_0.jpg 推定角度 300.00度
1/1 [==============================] - 0s 16ms/step
/content/drive/MyDrive/angle32x32/test2/60_0.jpg 推定角度 60.00度
1/1 [==============================] - 0s 19ms/step
/content/drive/MyDrive/angle32x32/test2/90_0.jpg 推定角度 90.00度
1/1 [==============================] - 0s 21ms/step
/content/drive/MyDrive/angle32x32/test2/330_0.jpg 推定角度 330.00度
1/1 [==============================] - 0s 16ms/step
/content/drive/MyDrive/angle32x32/test2/30_0.jpg 推定角度 30.00度
1/1 [==============================] - 0s 16ms/step
/content/drive/MyDrive/angle32x32/test2/210_0.jpg 推定角度 210.00度
1/1 [==============================] - 0s 15ms/step
/content/drive/MyDrive/angle32x32/test2/180_0.jpg 推定角度 180.00度
1/1 [==============================] - 0s 18ms/step
/content/drive/MyDrive/angle32x32/test2/150_0.jpg 推定角度 150.00度
1/1 [==============================] - 0s 16ms/step
/content/drive/MyDrive/angle32x32/

Anacondaの場合

In [15]:
test_list = glob.glob('angle32x32/test2/*.jpg')#Windows+Anacondaの場合
for f in test_list:
    raw = tf.io.read_file(f)
    img = tf.image.decode_image(raw, channels=1)
    img = img.numpy() / 255.0
    img = tf.expand_dims(img, axis=0) # バッチ次元の追加
    pred = model.predict(img) # 推論
    print(f, f'推定角度 {pred[0][0]:.2f}度')

Colab.の場合

In [20]:
test_list = glob.glob('/content/drive/MyDrive/angle32x32/test/*.jpg')#Colab.の場合
for f in test_list:
    raw = tf.io.read_file(f)
    img = tf.image.decode_image(raw, channels=1)
    img = img.numpy() / 255.0
    img = tf.expand_dims(img, axis=0) # バッチ次元の追加
    pred = model.predict(img) # 推論
    print(f, f'推定角度 {pred[0][0]:.2f}度')

1/1 [==============================] - 0s 18ms/step
/content/drive/MyDrive/angle32x32/test/25.jpg 推定角度 26.36度
1/1 [==============================] - 0s 16ms/step
/content/drive/MyDrive/angle32x32/test/15.jpg 推定角度 16.34度
1/1 [==============================] - 0s 15ms/step
/content/drive/MyDrive/angle32x32/test/5.jpg 推定角度 0.49度
1/1 [==============================] - 0s 15ms/step
/content/drive/MyDrive/angle32x32/test/10.jpg 推定角度 7.51度
1/1 [==============================] - 0s 19ms/step
/content/drive/MyDrive/angle32x32/test/20.jpg 推定角度 23.33度


Anacondaの場合

In [14]:
test_list = glob.glob('angle32x32/test/*.jpg')#Windows+Anacondaの場合
for f in test_list:
    raw = tf.io.read_file(f)
    img = tf.image.decode_image(raw, channels=1)
    img = img.numpy() / 255.0
    img = tf.expand_dims(img, axis=0) # バッチ次元の追加
    pred = model.predict(img) # 推論
    print(f, f'推定角度 {pred[0][0]:.2f}度')